# R02 · When will a brain result work for a new person?

**Reading seminar · Weeks 1–2.** Open the assigned papers alongside this notebook. Record your responses in a personal copy and bring the evidence sheet to discussion. AI can help explain unfamiliar terms; your reading of the source guides the interpretation.

**Method:** [three-pass reading guide](../../curriculum/papers/READING_METHOD.md). **Worksheet:** [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md). **Assignment:** [A1 figure brief](../../curriculum/coursework/PAPER_TO_EXPERIMENT.md#a1--first-pass-figure-brief).

---

### Seminar Framing: Population Association Estimands vs. Foundation-Model Transfer

Few controversies in modern neuroscience have generated more confusion—especially when summarized by AI assistants or social-media headlines—than the apparent collision between **statistical sample-size audits** and **self-supervised foundation models**.
- On one side, consortium studies of thousands of individuals ([Marek et al., 2022, `PD02`](../../curriculum/papers/design.md#pd02); followed up by [Kang et al., 2024, `PD06`](../../curriculum/papers/design.md#pd06)) demonstrated that cross-sectional Brain-Wide Association Studies (BWAS) linking resting-state MRI or cortical thickness to complex cognitive/psychiatric traits have small true effect sizes ($|r| \approx 0.01\text{--}0.10$), meaning studies of $N = 25\text{--}100$ people suffer from massive sampling variability, inflated winners'-curse effect sizes (Type M error), and sign reversals (Type S error).
- On the other side, modern 3D self-supervised foundation models ([BrainIAC, Tak et al., 2026, `PM03`](../../curriculum/papers/modeling.md#pm03); see also [MindEye2, Scotti et al., 2024, `PM06`](../../curriculum/papers/modeling.md#pm06)) report strong downstream performance when fine-tuned or linear-probed on cohorts with only dozens or a hundred labeled examples (or 1 hour of fMRI data per subject in `PM06`).

Does `BrainIAC` "refute" `Marek et al.`? **No—and understanding why is one of the most important conceptual milestones in this course.** In this seminar, you will learn to dissect the **estimand**, the **signal-to-noise ratio of the target phenotype**, the **information boundary between pretraining and downstream evaluation**, and the **hierarchy of generalization** (new scan of the same person $\to$ new person at the same hospital scanner $\to$ new person at an unseen hospital scanner, connected to [ComBat `PD07`](../../curriculum/papers/design.md#pd07) and [CPM `PM07`](../../curriculum/papers/modeling.md#pm07)).


## Read this pair first: Consortium Sample-Size Laws vs. Self-Supervised MRI Transfer

- **Essential Foundational Paper (`PD02`):** Marek et al., *Reproducible brain-wide association studies require thousands of individuals* (Nature, 2022; 603:654–660; including the 2022 open-access/date correction record). [Full paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC8991999/) · [PD02 coursework card](../../curriculum/papers/design.md#pd02).
- **Recent Frontier Method (`PM03`):** Tak et al., *A generalizable foundation model for analysis of human brain MRI*—BrainIAC (Nature Neuroscience, 2026). [Full paper](https://www.nature.com/articles/s41593-026-02202-6) · [PM03 coursework card](../../curriculum/papers/modeling.md#pm03).

**First-pass reading targets:**
1. **In `PD02` (Marek et al. BWAS):** Abstract, opening definition of BWAS (vs. within-subject task effects), **Figure 1e–h** (sampling variability funnel across subsample sizes $n = 25$ to $n = 3,928$ in ABCD, alongside **Figure 2** comparing size-matched subsamples across ABCD, HCP, and UK Biobank), **Figure 3** (statistical power, false negatives, Type S sign error, Type M magnitude inflation, and univariate replication), **Figure 4** (multivariate PCA + SVR/CCA in-sample overfitting vs. out-of-sample replication), and Discussion on why small-$n$ significant hits are systematically inflated.
2. **In `PM03` (BrainIAC):** Abstract, Results overview, **Figure 1** (`ViT-B` SimCLR contrastive pretraining on $32,015$ multiparametric 3D brain MRIs from $16$ pretraining datasets within a $48,965$-scan, $34$-dataset study pool, evaluated across $7$ downstream tasks), **Figure 2 & caption** (downstream fine-tuning performance across $10\%, 20\%, 40\%, 60\%, 80\%, \text{and } 100\%$ of labeled training data comparing BrainIAC, BrainSegFounder, MedicalNet, and Scratch), **Figure 3 & caption** (few-shot adaptation and frozen linear-probe evaluation), Methods on pretraining/data splits, and Discussion limitations.

### Side-by-Side Forensic Comparison (`PD02` vs. `PM03` & Generalization Strand Context)

| Analytical Dimension | `PD02` · *Marek et al. BWAS* (Nature, 2022) | `PM03` · *BrainIAC* (Tak et al., 2026) | Connections to `PD06`, `PD07`, `PM06`, `PM07` |
|---|---|---|---|
| **Primary Scientific Question & Estimand** | **Unbiased population parameter estimation & replicability:** How many independent individuals ($N$) are needed to reproducibly estimate cross-sectional associations ($r$ or multivariate weights) between brain MRI features (RSFC / cortical thickness) and cognitive/behavioral traits? | **Representation transfer & label efficiency:** Can self-supervised contrastive pretraining (SimCLR) on 32,000+ unannotated clinical/research structural brain MRIs learn a generalizable latent representation that improves downstream clinical prediction when labeled target data are scarce? | [`PD06` (Kang et al., 2024)](../../curriculum/papers/design.md#pd06) shows how widening covariate variance (oversampling extreme phenotypes) and separating between- vs. within-person longitudinal effects increase standardized effect sizes and BWAS replicability. |
| **Target Phenotype & Signal-to-Noise Regime** | **Subtle between-person behavioral/psychiatric traits** (e.g., cognitive ability, psychopathology scores in healthy/population cohorts) where true univariate brain-behavior correlations peak at $|r| \le 0.16$ (median $|r| \approx 0.01$) and measurement reliability is modest (`PD03`). | **Seven structural & clinical tasks spanning simple to challenging regimes:** MRI sequence classification, chronological Brain Age ($\text{MAE}$ in years), MCI vs. cognitively normal classification, glioma IDH mutation status, glioblastoma overall survival, time-since-stroke onset, and FLAIR glioma segmentation. | [`PM06` (MindEye2)](../../curriculum/papers/modeling.md#pm06) achieves 1-hour fMRI-to-image decoding by aligning *within-subject* visual responses into a shared multi-subject latent space on ultra-high-field 7T NSD data. |
| **What One Observation Represents** | **One independent participant** ($i \in \{1, \dots, N\}$) in cross-sectional population subsampling across ABCD ($N \approx 3,928$), HCP ($N \approx 900$), and UK Biobank ($N \approx 32,572$). | **Pretraining:** $32,015$ 3D volumetric structural MRI scans ($T_1\text{w}, T_1\text{CE}, T_2\text{w}, T_2\text{-FLAIR}$) across $16$ pretraining datasets (from a $48,965$-scan, $34$-dataset pool). **Downstream:** Labeled scans/patients split into train, validation, and internal/external test cohorts. | [`PD07` (ComBat)](../../curriculum/papers/design.md#pd07) models scanner-specific additive ($\gamma_{iv}$) and multiplicative ($\delta_{iv}$) batch effects that otherwise confound multi-site cohorts (`P03`). |
| **Role of Prior Data / Pretraining** | None (each subsample fits univariate correlations or multivariate PCA + SVR/CCA models from scratch on the sampled $N$ individuals and tests on an independent hold-out sample). | **Massive prior geometric/intensity regularizer:** 3D Vision Transformer (`ViT-B`) encoder (chosen after benchmarking against ResNet50 and MAE-SwinViT) is pretrained via SimCLR contrastive learning on $32,015$ unlabeled scans, reducing the effective degrees of freedom needed when fitting a linear probe or fine-tuning on $N_{\text{labeled}}$ downstream patients. | [`PM07` (CPM)](../../curriculum/papers/modeling.md#pm07) enforces strict fold-isolated edge screening when predicting behavior from connectomes without pretraining. |
| **Core Failure Mode Exposed or Audited** | **The Winner's Curse (Type M Inflation) & In-Sample Overfitting:** When $N=50$ and true $r=0.05$, the only associations that pass $p < 0.05$ (or uncorrected thresholds) are those with extreme random sampling noise ($|\hat{r}| \gtrsim 0.28\text{--}0.35$), which fail to replicate out-of-sample. | **Scratch Overfitting & Domain Shift in Low-Label Clinical Tasks:** Training a 3D CNN from scratch on 20–50 clinical scans memorizes noise or scanner artifacts; furthermore, even pretrained models must be audited on **external hospital cohorts** and under frozen vs. adapted regimes. | Even with `BrainIAC` pretraining, if your downstream **test set** contains only $N_{\text{test}} = 20$ patients, the confidence interval on your test AUC remains wide due to binomial sampling variance (`DS08`, `D04`)! |


## Separate the questions before comparing the papers

Before writing a single paragraph comparing `PD02` and `PM03`, complete this forensic decomposition table in your personal copy. **Do not assume that scans, participants, longitudinal visits, labeled fine-tuning examples, and unlabeled pretraining volumes count the same thing:**

| Forensic Audit Item | `PD02` · *Marek et al. BWAS* (2022) | `PM03` · *BrainIAC* (2026) |
|---|---|---|
| **1. Scientific or Clinical Target:** What exact relationship or prediction is being evaluated? | | |
| **2. Observation Unit ($N$):** What does one data point represent in the main figure? | | |
| **3. Modality & Phenotype SNR:** What MRI modality and what biological/behavioral outcome are linked? | | |
| **4. Total Data Used Before Evaluation:** Include unlabeled pretraining scans, feature screening, and labeled training sets. | | |
| **5. Held-Out Evaluation Data:** How is the test set separated (split-half cohort vs. external clinical site)? | | |
| **6. Adaptation Regime (if applicable):** Scratch fit vs. Frozen Linear Probe vs. Full Fine-Tuning. | | |
| **7. Exact Figure/Section Locator:** Where did you verify rows 1–6 in the primary text? | | |
| **8. Remaining Uncertainty:** What claim is *not* established by the inspected figure? | | |

### Forensic Figure Reading Guide (`PD02` Fig. 1, 3 & 4 vs. `PM03` Fig. 2 & 3)

1. **Reading `PD02` (Marek et al.) Figure 1e–h, Figure 3, and Figure 4:**
   - Look at the **funnel plots** in Figure 1e–h (in ABCD, with cross-cohort comparisons in Figure 2): the $x$-axis is subsample size $n$ (on a logarithmic scale from $n=25$ to $3,928$) and the $y$-axis is the observed brain-behavior correlation $r$. Why is the funnel so wide at $n=25$ (spanning roughly $r = -0.5$ to $+0.5$)? Because the standard error of Pearson's $r$ scales as $\text{SE}(r) \approx \frac{1 - r^2}{\sqrt{n - 2}} \approx \frac{1}{\sqrt{n}}$. At $n=25$, $\text{SE}(r) \approx 0.20$, which exceeds even the largest replicated univariate BWAS effect in the study ($|r| = 0.16$, with the top $1\%$ above $|r| = 0.06$ and median $|r| \approx 0.01$).
   - In **Figure 3** (univariate errors and replication) and **Figure 4** (multivariate PCA + SVR and CCA), trace what happens when a researcher applies a statistical significance threshold ($p < 0.05$ or Bonferroni $\alpha = 10^{-7}$) or fits a multivariate model at small $n$: only subsamples that randomly drew an extreme noise fluctuation cross the threshold (**Type M magnitude inflation**), and in Figure 4 in-sample multivariate correlations are strongly inflated (mean $r_{\text{in}} = 0.46$ vs. out-of-sample mean $r_{\text{out}} = 0.17$, a $63\%$ drop), with small-$n$ fits frequently failing statistical significance out of sample until sample sizes reach the thousands.
   - *Scope check:* Read the authors' explicit caveat in the Introduction and Discussion: this sample-size requirement applies to **between-person cross-sectional BWAS of complex behavioral and psychiatric traits**, whereas within-person experimental designs, longitudinal repeated-measures tracking, focal lesion studies, and group-average functional brain organization operate in substantially higher signal-to-noise regimes (a contrast that also extends to macroscopic structural targets such as chronological Brain Age in `PM03`).

2. **Reading `PM03` (BrainIAC) Figure 2 and Figure 3:**
   - Look at the **label-efficiency curves** in **Figure 2** (evaluating $10\%, 20\%, 40\%, 60\%, 80\%, \text{and } 100\%$ of labeled downstream training data across Fine-Tuned BrainIAC, BrainSegFounder, MedicalNet, and Scratch) alongside **Figure 3** (evaluating few-shot adaptation at $K = 1, 5, 10$ and frozen linear probing). Watch metric direction carefully: **lower is better** for Brain Age $\text{MAE}$, whereas **higher is better** for classification $\text{AUC}$ / balanced accuracy and segmentation $\text{Dice}$.
   - *Crucial distinction:* Why can BrainIAC improve downstream prediction when fine-tuned or linear-probed on cohorts with fewer than $200$ labeled patients? First, it is **not** starting from $N=200$—its 3D `ViT-B` encoder already learned structural priors from **$32,015$ pretraining scans** across $16$ datasets. Second, macroscopic structural targets such as chronological Brain Age or FLAIR tumor boundaries carry much stronger anatomical signal than subtle cross-sectional behavioral questionnaires ($|r| \approx 0.01$), although challenging clinical endpoints (such as IDH mutation status, GBM survival, and time-to-stroke) still exhibit moderate test AUCs ($\sim 0.65\text{--}0.80$) with wide bootstrap confidence intervals on small test cohorts.


## Original Assignment: Draw the Information Boundaries (The 32-Person / 128-Scan Audit)

Consider this invented supervision scenario:

> **The Scenario:** Your local clinical research project has recruited **32 patients**, each scanned **four times** across two months ($32 \times 4 = 128$ volumetric MRI scans total), to predict a clinical symptom severity score. A collaborator offers you a "pretrained 3D MRI encoder" trained on a public multi-institutional repository of 15,000 scans, plus access to a **future untouched cohort of 50 patients from a partner hospital** Across town (acquired on a different vendor's scanner). You do not yet have the subject/dataset manifest for the 15,000-scan pretraining repository, so you do not know whether your local institution's earlier public releases (which included some of your 32 patients) were scraped into the pretraining set.

### Step 1: Draw the Four Information Boxes & Place the Observations
Using Markdown boxes or a diagram, draw four strictly separated compartments:
1. **`[Box 1: Pretraining Corpus]`** (Unlabeled representation learning)
2. **`[Box 2: Downstream Task Fitting]`** (Training regression/classification weights)
3. **`[Box 3: Model Selection & Hyperparameter Tuning]`** (Choosing regularization $\alpha$, learning rate, or early stopping)
4. **`[Box 4: Final Confirmatory Evaluation]`** (Locked test set)

Now answer explicitly on your diagram:
- Where do the **4 repeated scans of Patient #1** go? Why must all 4 scans of Patient #1 sit inside the **same fold** (never 3 in Box 2 and 1 in Box 3/4)? What anatomical fingerprint would leak if you split by scan row (`P03`)?
- What exact metadata fields (e.g., BIDS dataset DOIs, site IDs, hashed subject GUIDs, acquisition dates) must you audit to rule out **pretraining-to-test patient overlap** between Box 1 and Boxes 3/4? Until you verify those manifests, how must you label that boundary (`[? Unknown Overlap]`)?
- What specific generalization question does **Box 4 (Partner Hospital Cohort)** test that an internal participant-grouped cross-validation within your 32 local patients **cannot** test (`PD07`, `P03`)?

### Step 2: Compare Three Adaptation Plans Under Fixed Constraints
Construct a three-row comparison for your 32-patient local cohort:
- **Plan A (Fresh Model from Scratch):** Fit a 3D CNN end-to-end on the 32 local patients (128 scans) from random initialization.
- **Plan B (Frozen Pretrained Representation + Regularized Linear Head):** Freeze all weights of the pretrained 3D encoder; extract a 512-dimensional embedding vector per scan and fit a Ridge regression head (`M04`, `M16`).
- **Plan C (Full End-to-End Fine-Tuning of Pretrained Weights):** Unfreeze all millions of parameters in the pretrained 3D encoder and update them using gradient descent on your 32 patients.

For each plan, state: (1) what parameters are fitted on your 32 local people, (2) what comparator must be held fixed, and (3) **one concrete way the plan could produce an impressive-looking training or cross-validation score while completely failing at the partner hospital**.


## Read a real result carefully & Synthesize the Dialectic

Choose one specific result panel from `PD02` (e.g., Fig. 1f/g or Fig. 3b/d) and one from `PM03` (e.g., Fig. 2 or Fig. 3 panels). For each panel, record:
- The exact **task and estimand**;
- The **observation unit** and sample size ($N_{\text{train}}$ vs. $N_{\text{test}}$);
- The **metric** and **direction of improvement**;
- How **uncertainty or sampling variability** is displayed (e.g., 95% confidence bands across random subsamples vs. standard deviation across runs/splits).

### 200-Word Dialectical Comparison
Write a rigorous 200-word synthesis answering:
> *“What structural problem in neuroimaging does Marek et al. (`PD02`) force me to confront, what capability does BrainIAC (`PM03`) provide, and what fundamental statistical or design problem does each paper leave unresolved by the other?”*

**Guardrails for your synthesis:**
- Do **not** claim that `BrainIAC` disproves `Marek et al.`, and do **not** claim that `Marek et al.` makes clinical transfer learning with $N < 1,000$ impossible.
- Contrast **true biological effect size / phenotype SNR** (behavioral questionnaire BWAS vs. structural pathology/age), **prior information** (fitting from scratch vs. 32,000-scan self-supervised pretraining), and **test-set precision** (why evaluating any model—even a foundation model—on a tiny test set of $N_{\text{test}} = 25$ people still yields huge uncertainty intervals on test performance!).
- Conclude by proposing **one concrete validation test** you would add for your own future research project, clearly labeled as **your proposal**.

**Deliverables for `R02` ([A1 Figure Brief](../../curriculum/coursework/PAPER_TO_EXPERIMENT.md#a1--first-pass-figure-brief)):**
1. Completed 8-row question-separation table;
2. The 4-box Information-Boundary diagram for the 32-person / 128-scan / partner-hospital scenario + 3-plan comparison;
3. One checked forensic figure row per paper + the 200-word dialectical synthesis;
4. Three technical questions to carry into Data Science, Research Design, and Modeling.


## Return after the fundamentals

This seminar directly motivates the statistical, design, and machine-learning core of the course:
- Study [DS05 · Missingness](../../notebooks/03_data_science/05_missingness.ipynb), [DS07 · Sampling](../../notebooks/03_data_science/07_sampling.ipynb), [DS08 · Bootstrap](../../notebooks/03_data_science/08_bootstrap.ipynb), and [DS15 · Selection bias](../../notebooks/03_data_science/15_selection.ipynb) (where you will simulate sampling funnels, winner's curse inflation, and participant-level bootstrap intervals);
- Study [D02 · Sampling, missingness, and units](../../notebooks/02_design/02_sampling_missingness_units.ipynb), [D03 · Reliability and measurement](../../notebooks/02_design/03_reliability_measurement.ipynb) (motivated by [Elliott et al. `PD03`](../../curriculum/papers/design.md#pd03) on how measurement noise attenuates observable correlations $r_{\text{obs}} = r_{\text{true}}\sqrt{\text{ICC}_X \cdot \text{ICC}_Y}$), and [D04 · Power and precision](../../notebooks/02_design/04_power_precision.ipynb);
- Revisit [PD06, the 2024 study-design follow-up by Kang et al.](../../curriculum/papers/design.md#pd06) before deciding what the 2022 BWAS headline means for extremity-enriched or longitudinal designs;
- Study [M04 · Nested cross-validation, sites, and harmonization](../../notebooks/04_modeling/04_nested_sites_harmonization.ipynb) (motivated by [ComBat `PD07`](../../curriculum/papers/design.md#pd07)), [M16 · Self-supervision and transfer](../../notebooks/04_modeling/16_selfsupervision_transfer.ipynb), [M18 · Calibration, explanation, and reproducibility](../../notebooks/04_modeling/18_calibration_explanation_reproducibility.ipynb), and Capstone [P03 · Cohort generalization](../../notebooks/05_projects/03_cohort_generalization.ipynb).

When you complete [A2 · Mechanism experiment](../../curriculum/coursework/PAPER_TO_EXPERIMENT.md#a2--mechanism-experiment) with `P03` or `M16`, you will run a controlled simulation of participant leakage, site-shortcut collapse, and frozen-vs-fine-tuned transfer—and use those exact numerical diagnostics to refine your `R02` evidence ledger.


## Tutor prompt and instructor check

> **Supervisory Tutor Prompt for `R02`:**
> "Use only the supplied paper excerpts and my 4-box information-boundary diagram. First, ask me what mathematical quantity is being estimated in Marek et al. (`PD02`) versus predicted in BrainIAC (`PM03`). Second, audit my diagram to check whether I confused: (a) 128 repeated scans with 32 independent people, (b) 32,000 unlabeled pretraining scans with downstream labeled sample size, or (c) internal participant-split validation with external unseen-hospital generalization. Cite exact source locations for paper-specific claims. Keep pretraining overlap marked as 'unknown' until manifest evidence is provided. Help me sharpen my three fundamentals questions without solving the optimization equations yet."

<details><summary>Instructor notes</summary>

### Grading & Discussion Guide for `R02`

1. **The 32-Person / 128-Scan Information-Boundary Audit:**
   - **Unit of Independence:** There are $N = 32$ independent people, not 128. Splitting randomly across the 128 scan rows places longitudinal visits of the same person in both train and validation/test sets. Because cortical morphology and baseline connectivity have high within-person finger-printing stability, a model can achieve near-perfect accuracy by memorizing subject identity (`P03`). All 4 scans of a person must be grouped together via `GroupKFold(groups=person_id)`.
   - **Pretraining Overlap Gate:** Because public neuroimaging repositories (ADNI, OASIS, IXI, ABIDE, BraTS, HCP, UKB) are routinely aggregated into foundation-model pretraining corpora (`PM03` aggregates 34 datasets in its study pool, with 16 used for pretraining), local public releases may already sit inside the pretrained weights. Learners must mark overlap **Unknown (`[?]`)** until dataset DOIs and subject IDs are cross-checked.
   - **Role of the Partner Hospital Cohort:** Internal `GroupKFold` on the 32 local patients only tests generalization to *new people on the same local scanner and recruitment protocol*. The untouched partner hospital cohort tests **out-of-distribution (OOD) transportability across scanner hardware, pulse sequence shifts, and demographic recruitment differences** (`PD07`, `P03`).

2. **Vulnerabilities of the Three Adaptation Plans (with $N = 32$ local patients):**
   - **Plan A (Scratch 3D CNN):** Millions of parameters fitted on 32 independent subjects $\to$ severe overfitting to scanner noise or spurious site/motion shortcuts; high variance across folds.
   - **Plan B (Frozen Pretrained Encoder + Ridge Head):** Prevents catastrophic overfitting of the 3D backbone, *but* fails if the pretrained features do not capture the specific subtle functional/clinical signal, or if scanner intensity shifts move embeddings out of domain.
   - **Plan C (Full End-to-End Fine-Tuning):** Updating all backbone weights on 32 people can distort the pretrained representation ("catastrophic forgetting" / feature distortion) and overfit to local batch confounds unless heavily regularized or adapted via low-rank/linear-probe-then-finetune schedules (`M16`).

3. **Synthesizing `PD02` (Marek) and `PM03` (BrainIAC):**
   - Reward students who recognize that **sample-size requirements depend on (1) the true effect size & measurement reliability of the target variable, (2) model complexity / prior pretraining constraints, and (3) whether we are estimating discovery associations across the whole brain or transferring macroscopic structural features**. Neither paper invalidates the other; together they define modern study design.

</details>

Next: [R03 analysis choices and benchmark claims](03_why_evidence_audits.ipynb).


<!-- agentic-guide -->
### Agentic Deliverable Supervision (R02 · macOS Goose + Ollama: Qwen & Gemma)

**Deliverable focus:** Paired evidence audit comparing BWAS sample-size replication curves (`PD02` Marek et al.) against foundation-model low-label transfer (`PM03` BrainIAC).

#### 1. Suggestive Prompt (in the spirit of what to ask — adapt, do not copy verbatim)
> Ask Goose/Gemma in the spirit of: *"Compare the observation unit, target variable, and validation regime in Marek et al. (`PD02` Figures 1–2) versus BrainIAC (`PM03` Figures 1–2). Help me audit why a noisy cross-sectional behavioral correlation requires thousands of participants while a structural pathology/brain-age task can adapt with far fewer labels—and flag any claim that pretraining eliminates sampling variability."*

#### 2. What to Look For & How to Trace the Error Back Down
- **Immediate Red Flag:** The AI claims that self-supervised MRI pretraining (`PM03`) 'solves' the Marek et al. (`PD02`) small-sample replication crisis for arbitrary behavioral traits.
- **Traceback & Fix:** Trace the signal-to-noise ratio and target reliability ($\text{ICC}$) in both papers: `PD02` evaluates weak univariate/multivariate resting-state associations with noisy psychometric questionnaires ($|r| \approx 0.05\text{–}0.15$), whereas `PM03` evaluates structural T1/FLAIR tasks with strong anatomical signatures. Separate *representation quality* from *target measurement noise*.

#### 3. Expected Outcome Range & Why It Must Look That Way
**Expected Range:** A paired comparison table contrasting true population effect sizes ($|r| \lesssim 0.15$ in BWAS vs. large structural effects in clinical MRI) and small-$N$ inflation ($2\times\text{–}5\times$ Type M exaggeration at $N = 25\text{–}50$ under $p < 0.05$ thresholding). **Why:** Sampling variance of a correlation scales as $\approx (1 - r^2)^2 / (N - 3)$ and is attenuated by $\sqrt{\text{ICC}_X \cdot \text{ICC}_Y}$; pretraining regularizes the feature weights $\hat{w}$ on $X$, but cannot reduce sampling noise or unreliability in the outcome $Y$ on a small test cohort.
<!-- /agentic-guide -->
